# 04 · Publish Scorecard Bundle & Eventhouse DDL

Export multi-run JSON bundles for the standalone scorecard preview app (`app_sample`) and preview the generated Eventhouse KQL DDL (`setup_script`). Defaults safely to `MOCK = True` so no Eventhouse ingestion occurs unless explicitly enabled.

In [ ]:
import json
import pathlib
import sys

repo = pathlib.Path.cwd()
while repo != repo.parent and not (repo / "src" / "cfeval").exists():
    repo = repo.parent
if (repo / "src" / "cfeval").exists() and str(repo / "src") not in sys.path:
    sys.path.insert(0, str(repo / "src"))

from cfeval.cli import app_sample, simulate_runs
from cfeval.config import Settings
from cfeval.kqlgen import setup_script

MOCK = True
PUBLISH_TO_EVENTHOUSE = False
out_root = repo / "build" / "notebook-runs"
settings = Settings.load(env_file=None, output_dir=str(out_root))

In [ ]:
# Generate a 3-run progression (baseline -> improving -> tuned) locally
outcomes = simulate_runs(settings, runs=3, days=3, seed=7, publish=(PUBLISH_TO_EVENTHOUSE and not MOCK))
bundle = app_sample(outcomes)

bundle_path = out_root / "scorecard-sample.json"
bundle_path.write_text(json.dumps(bundle, indent=2, default=str), encoding="utf-8")
print(f"Wrote scorecard bundle ({bundle_path.stat().st_size} bytes) with {len(bundle['tables']['EvalRun'])} runs.")

In [ ]:
# Preview the first 25 lines of the Eventhouse KQL DDL generated from schema.py
ddl_lines = setup_script(settings.eventhouse_database).splitlines()
print("\n".join(ddl_lines[:25]))